# Credit Risk — Individual PD Modeling

**Goal:** estimate the probability that an active loan enters default **next month**.  
**Data:** synthetic loan-month records from the EDA notebook.  
**Method:** compare classifiers on a validation period, then evaluate the strongest candidates on a final untouched test period.

Run this notebook after `01_credit_risk_eda_simple_en.ipynb`. Place `pd_dataset.csv` in this folder or its parent. Install packages if needed:

```bash
pip install pandas numpy matplotlib scikit-learn xgboost lightgbm catboost imbalanced-learn joblib
```

## 1. Load data

We use the labeled dataset prepared during EDA; the target is **default in the following month**.

In [ ]:
from pathlib import Path
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                              ExtraTreesClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score, log_loss)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier

# Project paths: notebooks work when Jupyter starts from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.2})


In [ ]:
source = PROJECT_ROOT / "data/processed/pd_dataset.csv"
if not source.exists():
    raise FileNotFoundError("Run the EDA notebook first to create data/processed/pd_dataset.csv")

df = pd.read_csv(source, parse_dates=["reference_month", "origination_date"])
df = df.sort_values(["reference_month", "loan_id"]).reset_index(drop=True)
print(f"Observations: {len(df):,} | Next-month defaults: {df.default_next_month.sum():,} "
      f"({df.default_next_month.mean():.2%})")
display(df.head(3))


## 2. Time-based split

The last 6 labeled months form the **test set**; the previous 6 months are for **model selection**. No random split.

In [ ]:
months = sorted(df['reference_month'].unique())
assert len(months) >= 18, 'Not enough monthly observations for this split.'

train = df[df.reference_month.isin(months[:-12])].copy()
valid = df[df.reference_month.isin(months[-12:-6])].copy()
test = df[df.reference_month.isin(months[-6:])].copy()

split_summary = pd.DataFrame({
    'sample': ['Train', 'Validation', 'Test'],
    'rows': [len(train), len(valid), len(test)],
    'default_rate': [p.default_next_month.mean() for p in [train, valid, test]],
    'from': [p.reference_month.min().date() for p in [train, valid, test]],
    'to': [p.reference_month.max().date() for p in [train, valid, test]],
})
display(split_summary.style.format({'default_rate': '{:.2%}'}))

## 3. Features and preprocessing

Use only information known at the **end of the reference month**. Categorical values are one-hot encoded; numerical values are standardized.

In [ ]:
categorical_features = ['segment', 'rating']
numeric_features = ['original_value', 'annual_int_rate', 'months_term',
                    'outstanding_balance', 'days_past_due', 'months_on_book']
features = categorical_features + numeric_features
target = 'default_next_month'

preprocessor = ColumnTransformer([
    ('category', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), categorical_features),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_features)
])

# Fit preprocessing on TRAIN ONLY.
X_train = preprocessor.fit_transform(train[features])
X_valid = preprocessor.transform(valid[features])
y_train, y_valid = train[target], valid[target]

print('Transformed features:', X_train.shape[1])

## 4. Candidate classifiers

We test **19 models/configurations**, including a no-skill baseline. These are reasonable starting settings—not hyperparameter-optimized models.

In [ ]:
models = {
    '01 Dummy (prior)': DummyClassifier(strategy='prior'),
    '02 Logistic Regression': LogisticRegression(max_iter=1000),
    '03 Logistic (balanced)': LogisticRegression(max_iter=1000, class_weight='balanced'),
    '04 SGD Logistic': SGDClassifier(loss='log_loss', alpha=0.0001, max_iter=2000, random_state=42),
    '05 LDA': LinearDiscriminantAnalysis(),
    '06 QDA': QuadraticDiscriminantAnalysis(reg_param=0.2),
    '07 Gaussian NB': GaussianNB(),
    '08 Decision Tree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=100, random_state=42),
    '09 Random Forest': RandomForestClassifier(n_estimators=120, max_depth=9, min_samples_leaf=30, n_jobs=2, random_state=42),
    '10 Extra Trees': ExtraTreesClassifier(n_estimators=120, max_depth=9, min_samples_leaf=30, n_jobs=2, random_state=42),
    '11 HistGradientBoosting': HistGradientBoostingClassifier(max_iter=110, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=100, random_state=42),
    '12 Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=42),
    '13 AdaBoost': AdaBoostClassifier(n_estimators=90, learning_rate=0.06, random_state=42),
    '14 Bagging Trees': BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5, min_samples_leaf=100), n_estimators=25, max_samples=0.8, n_jobs=2, random_state=42),
    '15 XGBoost': XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.045, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', n_jobs=2, random_state=42),
    '16 LightGBM': LGBMClassifier(n_estimators=140, num_leaves=11, learning_rate=0.05, min_child_samples=100, verbosity=-1, n_jobs=2, random_state=42),
    '17 CatBoost': CatBoostClassifier(iterations=150, depth=4, learning_rate=0.05, l2_leaf_reg=5, verbose=False, thread_count=2, random_seed=42),
    '18 MLP': MLPClassifier(hidden_layer_sizes=(20,), alpha=1.0, early_stopping=True, max_iter=130, random_state=42),
    '19 Balanced Random Forest': BalancedRandomForestClassifier(n_estimators=80, max_depth=8, min_samples_leaf=15, n_jobs=2, random_state=42),
}
print(f'{len(models)} candidates ready')

## 5. Validation benchmark

**Average Precision (AP)** ranks models because defaults are rare. **ROC-AUC** checks ranking and **Brier Score** evaluates the predicted probabilities (lower is better).

In [ ]:
def score(y_true, prob):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(prob, dtype=float)
    top_n = max(1, int(np.ceil(len(y) * 0.10)))
    top = np.argsort(-p)[:top_n]
    return {
        'AP': average_precision_score(y, p),
        'ROC_AUC': roc_auc_score(y, p),
        'Brier': brier_score_loss(y, p),
        'Precision@10%': y[top].mean(),
        'Recall@10%': y[top].sum() / y.sum(),
    }

rows = []
for name, estimator in models.items():
    started = time.perf_counter()
    try:
        estimator.fit(X_train, y_train)
        prob = estimator.predict_proba(X_valid)[:, 1]
        rows.append({'Model': name, **score(y_valid, prob),
                     'Time (s)': round(time.perf_counter() - started, 1)})
        print('OK ', name)
    except Exception as exc:
        print('FAILED', name, str(exc)[:120])

validation_results = pd.DataFrame(rows).sort_values('AP', ascending=False).reset_index(drop=True)
validation_results.to_csv(RESULTS_DIR / "pd_validation_results.csv", index=False)
display(validation_results.round(4))

In [ ]:
ax = validation_results.sort_values('AP').plot.barh(
    x='Model', y='AP', legend=False, figsize=(10, 7), title='Validation — Average Precision')
ax.set_xlabel('Average Precision (higher is better)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 6. Independent test

Select the **top 5 by validation AP** and compare them against the dummy baseline on the untouched test months. Refit using Train + Validation.

In [ ]:
train_valid = pd.concat([train, valid], ignore_index=True)
final_preprocessor = clone(preprocessor)
X_train_valid = final_preprocessor.fit_transform(train_valid[features])
X_test = final_preprocessor.transform(test[features])
y_train_valid, y_test = train_valid[target], test[target]

finalists = validation_results['Model'].head(5).tolist()
if '01 Dummy (prior)' not in finalists:
    finalists.append('01 Dummy (prior)')

holdout_rows = []
holdout_probs = {}
trained = {}
for name in finalists:
    estimator = models[name]
    estimator.fit(X_train_valid, y_train_valid)
    p = estimator.predict_proba(X_test)[:, 1]
    holdout_probs[name] = p
    trained[name] = estimator
    holdout_rows.append({'Model': name, **score(y_test, p)})

holdout_results = pd.DataFrame(holdout_rows)
holdout_results.to_csv(RESULTS_DIR / "pd_holdout_results.csv", index=False)
display(holdout_results.round(4))

winner_name = validation_results.iloc[0]['Model']
print('Selected model (by validation AP):', winner_name)
print('Independent test AP:', round(float(holdout_results.set_index('Model').loc[winner_name, 'AP']), 4))

## 7. How useful is the selected model?

The **Precision–Recall curve** shows the trade-off when changing the decision threshold. The **calibration plot** compares predicted probabilities with observed default rates.

In [ ]:
plt.figure(figsize=(8, 5))
for name in finalists:
    if name == '01 Dummy (prior)':
        continue
    precision, recall, _ = precision_recall_curve(y_test, holdout_probs[name])
    plt.plot(recall, precision, label=name.split(' ', 1)[1], alpha=0.85)
plt.axhline(y_test.mean(), linestyle=':', color='black', label='Default prevalence')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Holdout — Precision–Recall curves')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
winner_prob = holdout_probs[winner_name]
observed, predicted = calibration_curve(y_test, winner_prob, n_bins=8, strategy='quantile')

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], '--', label='Perfect calibration')
ax.plot(predicted, observed, 'o-', label='Selected model')
ax.set(xlabel='Mean predicted probability', ylabel='Observed default rate',
       title='Holdout — PD calibration')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Save predictions and the selected model

One row is a loan observed at a specific month, not necessarily a unique borrower. Saving the preprocessor preserves the feature transformations.

In [ ]:
predictions = test[['loan_id', 'reference_month', 'default_next_month']].copy()
predictions['predicted_pd'] = winner_prob
predictions.to_csv(RESULTS_DIR / "pd_holdout_predictions.csv", index=False)

joblib.dump({
    'preprocessor': final_preprocessor,
    'classifier': trained[winner_name],
    'features': features,
    'model_name': winner_name,
    'horizon': 'next month'
}, RESULTS_DIR / "pd_best_model.joblib")

display(predictions.sort_values('predicted_pd', ascending=False).head(10))

### Interpretation and limitations

- **Winner:** selected by validation AP, not by test results. Check test **Brier** before trusting numerical PDs.
- **Synthetic-data caveat:** DPD is strongly associated with next-month default. Real portfolios require external and out-of-time validation, stability checks, and calibration.
- **Next:** explain risk drivers, evaluate PD without `days_past_due` as a sensitivity check, and then estimate expected losses using PD × LGD × EAD.
